In [1]:
import sys
import os
sys.path.append('/Users/mariana/Documents/projects/Huawei/survan')
sys.path.append('/home/mvargas/code/Huawei/survan')

from lambda_cox import LambdaSA
from utils import get_churn_lastfm_dataset_months, get_churn_kkbox, get_targets_and_masks, train_test_split
import yaml
import jax
import jax.numpy as jnp
import haiku as hk
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
# path = '/Users/mariana/Documents/projects/Huawei/SurvanData/lastfm-dataset-1K'
data_path = '/home/mvargas/code/Huawei/SurvanData/kkbox-churn-prediction-challenge/kkbox'

In [3]:
df_logs = pd.read_feather(os.path.join(data_path, 'logs_filtered_preprocessed.feather'))
df_events = pd.read_feather(os.path.join(data_path, 'survival_preprocessed.feather'))
ts = df_events.time
cs = df_events.event
horizon = np.max(ts)

In [4]:
horizon

45

In [5]:
common_index = df_logs.index.intersection(df_events.index)

In [6]:
len(common_index)

1368900

In [7]:
len(df_logs)

5551586

In [8]:
len(df_events)

1368900

In [9]:
df_logs.columns

Index(['num_25', 'num_50', 'num_75', 'num_985', 'num_100', 'num_unq',
       'log_minutes', 'msno'],
      dtype='object')

In [10]:
df_events.columns

Index(['msno', 'time', 'event'], dtype='object')

In [12]:
ids = pd.unique(df_events.msno)

In [43]:
chunk = df_events.msno.sample(136890)
max_msno = df_logs.groupby('msno').size().idxmax()
chunk = pd.concat([chunk, pd.Series([max_msno])], ignore_index=True)

In [44]:
chunk.shape

(136891,)

In [45]:
chunk_logs = df_logs[df_logs['msno'].isin(chunk)]
chunk_evnts = df_events[df_events['msno'].isin(chunk)]

In [46]:
print(len(chunk_logs))
print(len(chunk_evnts))

555433
136890


In [47]:
chunk_logs.groupby('msno').size().max()

45

In [50]:
chunk_evnts.event.sum()/len(chunk_evnts)

0.414376506684199

In [51]:
chunk_logs_pth = os.path.join(data_path, 'chunk_logs_filtered_preprocessed.feather')
chunk_evts_pth = os.path.join(data_path, 'chunk_survival_preprocessed.feather')
chunk_logs.to_feather(chunk_logs_pth)
chunk_evnts.to_feather(chunk_evts_pth)

In [52]:
chunk_logs_pth

'/home/mvargas/code/Huawei/SurvanData/kkbox-churn-prediction-challenge/kkbox/chunk_logs_filtered_preprocessed.feather'